In [0]:
from pyspark.sql import Row

data = [
    Row(click_time="2018-01-01 11:00:00", user_id="u1"),
    Row(click_time="2018-01-01 12:00:00", user_id="u1"),
    Row(click_time="2018-01-01 13:00:00", user_id="u1"),
    Row(click_time="2018-01-01 13:00:00", user_id="u1"),
    Row(click_time="2018-01-01 14:00:00", user_id="u1"),
    Row(click_time="2018-01-01 15:00:00", user_id="u1"),
    Row(click_time="2018-01-01 11:00:00", user_id="u2"),
    Row(click_time="2018-01-02 11:00:00", user_id="u2"),
]

d = spark.createDataFrame(data)
display(d)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, collect_list, struct

spark = SparkSession.builder.appName("PersonItemWeight").getOrCreate()

data = [
    ("alice", "carrot", 1),
    ("bob", "banana", 3),
    ("alice", "tomato", 4),
    ("bob", "carrot", 2),
    ("charlie", "banana", 5),
    ("bob", "apple", 1),
    ("alice", "tomato", 2),
    ("charlie", "carrot", 3)
]

cols = ["name", "item", "weight"]

df = spark.createDataFrame(data, cols)
df=df.groupBy("name","item").agg(sum("weight").alias("weight"))
df=df.groupBy("name").agg(collect_list(struct("item","weight")).alias("items"))
display(df.orderBy("name"))

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("CookbookTitles").getOrCreate()

titles_data = [
    (1, 'Scrambled eggs'),
    (2, 'Fondue'),
    (3, 'Sandwich'),
    (4, 'Tomato soup'),
    (6, 'Liver'),
    (11, 'Fried duck'),
    (12, 'Boiled duck'),
    (15, 'Baked chicken')
]
titles_columns = ["page_number", "title"]

titles_df = spark.createDataFrame(titles_data, titles_columns)

left_pages_df = titles_df.filter(col("page_number") % 2 == 0).select("page_number", "title")
right_pages_df = titles_df.filter(col("page_number") % 2 != 0).select("page_number", "title")


left_pages_df = left_pages_df.withColumnRenamed("page_number", "left_page_number").withColumnRenamed("title", "left_title")
right_pages_df = right_pages_df.withColumnRenamed("page_number", "right_page_number").withColumnRenamed("title", "right_title")
result_df = left_pages_df.join(right_pages_df, left_pages_df.left_page_number + 1 == right_pages_df.right_page_number, "left") \
    .select("left_page_number", "left_title", "right_title")

result_df.show()

In [0]:
df = spark.createDataFrame([(1,), (2,), (3,)], ["id"])
output_df = df.selectExpr("explode(sequence(1, id)) as id")


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, round

famous_data = [
    (1, 2), (1, 3), (2, 4), (5, 1), (5, 3),
    (11, 7), (12, 8), (13, 5), (13, 10),
    (14, 12), (14, 3), (15, 14), (15, 13)
]

columns = ["user_id", "follower_id"]
df_famous = spark.createDataFrame(famous_data, columns)
distinct_users = df_famous.select("user_id").union(df_famous.select("follower_id")).distinct()
total_users = distinct_users.count()
followers_count=df_famous.groupBy("user_id").agg(countDistinct("follower_id").alias("follower_count"))
famous_percentage = followers_count.withColumn(
    "famous_percentage", 
    round((col("follower_count") / total_users), 2) * 100
)
display(famous_percentage)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc
from pyspark.sql.functions import rank, dense_rank, row_number, col
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("OscarWins").getOrCreate()

nominee_data = [
    ('Jennifer Lawrence', 'P562566', 'Drama', '1990-08-15', 755),
    ('Jonah Hill', 'P418718', 'Comedy', '1983-12-20', 747),
    ('Anne Hathaway', 'P292630', 'Drama', '1982-11-12', 744),
    ('Jennifer Hudson', 'P454405', 'Drama', '1981-09-12', 742),
    ('Rinko Kikuchi', 'P475244', 'Drama', '1981-01-06', 739)
]

oscar_data = [
    (2008, 'actress in a leading role', 'Anne Hathaway', 'Rachel Getting Married', 0, 77),
    (2012, 'actress in a supporting role', 'Anne HathawayLes', 'Mis_rables', 1, 78),
    (2006, 'actress in a supporting role', 'Jennifer Hudson', 'Dreamgirls', 1, 711),
    (2010, 'actress in a leading role', 'Jennifer Lawrence', 'Winters Bone', 1, 717),
    (2012, 'actress in a leading role', 'Jennifer Lawrence', 'Silver Linings Playbook', 1, 718),
    (2011, 'actor in a supporting role', 'Jonah Hill', 'Moneyball', 0, 799),
    (2006, 'actress in a supporting role', 'Rinko Kikuchi', 'Babel', 0, 1253)
]

columns_nominee = ["name", "amg_person_id", "top_genre", "birthday", "id"]

columns_oscar = ["year", "category", "nominee", "movie", "winner", "id"]

df_nominee = spark.createDataFrame(nominee_data, columns_nominee)
df_oscar = spark.createDataFrame(oscar_data, columns_oscar)

final_df=df_nominee.join(df_oscar,df_nominee.name==df_oscar.nominee,"inner")
final_df=final_df.select('name','top_genre','winner').filter(col('winner')==1)
final_df=final_df.groupBy('name','top_genre').agg(sum(col('winner')).alias('winner'))
final_df=final_df.withColumn('rank',rank().over(Window.partitionBy('top_genre','name').orderBy('winner')))
display(final_df.filter(final_df.rank==1))

In [0]:
import datetime
from pyspark.sql.types import StructType, StructField, TimestampType, LongType, StringType
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("TotalInTime") \
    .getOrCreate()

input_data = [
    (11114, datetime.datetime.strptime('08:30:00.00', "%H:%M:%S.%f"), "I"),
    (11114, datetime.datetime.strptime('10:30:00.00', "%H:%M:%S.%f"), 'O'),
    (11114, datetime.datetime.strptime('11:30:00.00', "%H:%M:%S.%f"), 'I'),
    (11114, datetime.datetime.strptime('15:30:00.00', "%H:%M:%S.%f"), 'O'),
    (11115, datetime.datetime.strptime('09:30:00.00', "%H:%M:%S.%f"), 'I'),
    (11115, datetime.datetime.strptime('17:30:00.00', "%H:%M:%S.%f"), 'O')
]

input_schema = StructType([
    StructField('emp_id', LongType(), True),
    StructField('punch_time', TimestampType(), True),
    StructField('flag', StringType(), True)
])

df = spark.createDataFrame(data=input_data, schema=input_schema)

window_def = Window.partitionBy('emp_id').orderBy(col('punch_time'))

df = df.withColumn('prev_time', lag(col('punch_time')).over(window_def))

df = df.withColumn('time_diff', (col('punch_time').cast('long') - col('prev_time').cast('long'))/3600)
display(df)
df = df.groupBy('emp_id').agg(sum(when(col('flag') == 'O', col('time_diff')).otherwise(0)).alias('total_time'))

df.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc
from pyspark.sql.functions import rank, dense_rank, row_number, col
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("OscarWins").getOrCreate()

nominee_data = [
    ('Jennifer Lawrence', 'P562566', 'Drama', '1990-08-15', 755),
    ('Jonah Hill', 'P418718', 'Comedy', '1983-12-20', 747),
    ('Anne Hathaway', 'P292630', 'Drama', '1982-11-12', 744),
    ('Jennifer Hudson', 'P454405', 'Drama', '1981-09-12', 742),
    ('Rinko Kikuchi', 'P475244', 'Drama', '1981-01-06', 739)
]

oscar_data = [
    (2008, 'actress in a leading role', 'Anne Hathaway', 'Rachel Getting Married', 0, 77),
    (2012, 'actress in a supporting role', 'Anne HathawayLes', 'Mis_rables', 1, 78),
    (2006, 'actress in a supporting role', 'Jennifer Hudson', 'Dreamgirls', 1, 711),
    (2010, 'actress in a leading role', 'Jennifer Lawrence', 'Winters Bone', 1, 717),
    (2012, 'actress in a leading role', 'Jennifer Lawrence', 'Silver Linings Playbook', 1, 718),
    (2011, 'actor in a supporting role', 'Jonah Hill', 'Moneyball', 0, 799),
    (2006, 'actress in a supporting role', 'Rinko Kikuchi', 'Babel', 0, 1253)
]

columns_nominee = ["name", "amg_person_id", "top_genre", "birthday", "id"]

columns_oscar = ["year", "category", "nominee", "movie", "winner", "id"]

df_nominee = spark.createDataFrame(nominee_data, columns_nominee)
df_oscar = spark.createDataFrame(oscar_data, columns_oscar)

df_test = df_nominee.withColumn(
    "rank",
    rank().over(Window.partitionBy("name").orderBy("name"))
)
display(df_test)

In [0]:
data = [
    ("hello world",),
    ("hello PySpark",),
    ("PySpark is fun",),
    ("hello world again",)
]
df = spark.createDataFrame(data, ["text"])
display(df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, col

# Split lines into words and explode them into rows
words_df = df.select(explode(split(col("text"), " ")).alias("word"))
display(words_df)
# Count occurrences of each word
count_df = words_df.groupBy("word").count()

# Sort by count in descending order
result_df = count_df.orderBy(col("count").desc())

display(result_df)
